In [6]:
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [8]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [9]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "google_genai:gemini-3.1-flash-lite",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [10]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [11]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='8a2147d8-81e8-4a32-ba97-64058547653c'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'call_15999638565826205794': 'EnMKcQFpFH0T0wzcuUc+8VXNfrQF8OXYUytzQGGmvHiHMeZ0j0ztkUYIonxZII/YjXhnLJngzX41mtnB3A7I/gFWcb3F3H1XdDkDVmX+PImtzSQ92IQIHnm3qMIqb1oyK89ISnhH1+z6zgZKzbunuVcbxm47'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cfd4-a8f5-7d53-b729-347bdafde486-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_15999638565826205794', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 69, 'output_tokens': 22, 'to

In [12]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='45a11a86-932a-4ed4-86dc-10adc0347280'),
              AIMessage(content=[{'type': 'text', 'text': "I'm doing well, thank you for asking! How are you doing today?", 'extras': {'signature': 'EnMKcQFpFH0Ta80+TIDRk9T8/VZIouJR7KuD2N0H7WW6vI+5yE64Y/AZL77+o9BcIK0Q+R6jBWXtXHYjWuwZQfbWTZ1qhu60QzQNhoqgagaiu6RGoBi/aFBmtXM5Fn89p6BvKNWQM6mEHgIfN/oi3BKG0xBs'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cfd4-af5f-7242-9146-1074ea74e16b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 70, 'output_tokens': 17, 'total_tokens': 87, 'input_token_details': {'cache_read': 0}})]}


## Read state

In [13]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "google_genai:gemini-3.1-flash-lite",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [14]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='d907c8e8-72a5-4aaf-93fe-686f7f528d92'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'call_3866530930383225687': 'EnMKcQFpFH0Ts+PFvAg41sqemmK1vdAiRIWCMlDnodDcOkIGztduOrNaDiGlDoL9qZYyJpdh/YdWzby3N8AesKXEDVr85WDLSzERfFNcx3Z1xT2xGCwJhhaULQIbBMYy0nNMkb26fp4V16XCWCXDJGo9VfuC'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cfd4-b31d-7073-bcc7-d2f67ec1a1de-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_3866530930383225687', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 22, 'tota

In [15]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='d907c8e8-72a5-4aaf-93fe-686f7f528d92'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'call_3866530930383225687': 'EnMKcQFpFH0Ts+PFvAg41sqemmK1vdAiRIWCMlDnodDcOkIGztduOrNaDiGlDoL9qZYyJpdh/YdWzby3N8AesKXEDVr85WDLSzERfFNcx3Z1xT2xGCwJhhaULQIbBMYy0nNMkb26fp4V16XCWCXDJGo9VfuC'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cfd4-b31d-7073-bcc7-d2f67ec1a1de-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_3866530930383225687', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 22, 'tota